## PyTorch Fundamentals

### PyTorch Tensors 

In [1]:
import torch

In [2]:
X = torch.tensor(
    [
        [1.0, 4.0, 7.0],
        [2.0, 3.0, 6.0]
    ],
)

In [3]:
X

tensor([[1., 4., 7.],
        [2., 3., 6.]])

In [4]:
print(X.shape)
print(X.dtype)

torch.Size([2, 3])
torch.float32


In [5]:
print(X[0, 1])
print(X[:,1])

tensor(4.)
tensor([4., 3.])


In [6]:
10 * (X + 1.0)

tensor([[20., 50., 80.],
        [30., 40., 70.]])

In [7]:
print(X.exp())
print(X.mean())
print(X.max(dim=0)) # note: dim 0 means across dim 0 i.e. across rows i.e. max of all the cols

tensor([[   2.7183,   54.5981, 1096.6332],
        [   7.3891,   20.0855,  403.4288]])
tensor(3.8333)
torch.return_types.max(
values=tensor([2., 4., 7.]),
indices=tensor([1, 0, 0]))


In [8]:
X @ X.T

tensor([[66., 56.],
        [56., 49.]])

Converting a tensor into a numpy array.

In [9]:
import numpy as np
X.numpy()

array([[1., 4., 7.],
       [2., 3., 6.]], dtype=float32)

In [10]:
torch.tensor(np.array(
    [
        [1.0, 4.0, 7.0],
        [2.0, 3.0, 6.0]
    ]
))

tensor([[1., 4., 7.],
        [2., 3., 6.]], dtype=torch.float64)

The default precision for floats is 32 bits in PyTorch, whereas it’s 64 bits in NumPy. It’s generally better to use 32 bits in deep learning. So when calling the torch.tensor() function
to convert a NumPy array to a tensor, it’s best to specify dtype=torch.float32.

In [11]:
torch.tensor(np.array(
    [
        [1.0, 4.0, 7.0],
        [2.0, 3.0, 6.0]
    ]
), dtype=torch.float32)

tensor([[1., 4., 7.],
        [2., 3., 6.]])

You can also modify a tensor in place using indexing and slicing, as with a NumPy array.

In [12]:
X[:, 1] = -99
X

tensor([[  1., -99.,   7.],
        [  2., -99.,   6.]])

In-place operations: abs_(), sqrt_(), zero_(), relu_()  
Regular operations: abs(), sqrt(), zero(), relu()

In [13]:
X.abs_()
X

tensor([[ 1., 99.,  7.],
        [ 2., 99.,  6.]])

### Hardware Acceleration


In [14]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0))

True
1
0
NVIDIA GeForce RTX 4060 Laptop GPU
_CudaDeviceProperties(name='NVIDIA GeForce RTX 4060 Laptop GPU', major=8, minor=9, total_memory=8187MB, multi_processor_count=24, uuid=a682ab30-74b5-8cfa-b6d0-884254cbea06, pci_bus_id=1, pci_device_id=0, pci_domain_id=0, L2_cache_size=32MB)


In [15]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

device

'cuda'

Lets create a tensor on the GPU. One option is to create tensor on the CPU and then copy it to the GPU using the to() method.


In [16]:
M = torch.tensor(
    [
        [1., 2., 3.],
        [4., 5., 6.]
    ]
)
M = M.to(device)

M.device

device(type='cuda', index=0)

Another option is to create tensor directly on the GPU using device argument.


In [17]:
M = torch.tensor(
    [
        [1., 2., 3.],
        [4., 5., 6.]
    ], 
    device=device
)

M.device

device(type='cuda', index=0)

Once the tensor is on the GPU, we can run operations on it normally, and they will all take place on the GPU:

In [18]:
R = M @ M.T
R

tensor([[14., 32.],
        [32., 77.]], device='cuda:0')

Let’s run a little test to compare the speed of a matrix multiplication running on the CPU versus the GPU.

In [19]:
M = torch.rand((1000, 1000))
%timeit M @ M.T

7.98 ms ± 170 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [20]:
M = torch.rand((1000, 1000), device=device)
%timeit M @ M.T

308 μs ± 6.66 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


GPU gave around 25 times than the CPU.

## Autograd

In [48]:
x = torch.tensor(5.0 , requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [49]:
f.backward() 
# computes derivative of f wrt every variable it depends on (just x here)
# and stores the derivative in .grad of the respective variables it depends on
x.grad

tensor(10.)

Let's try for x1, x2 and f1, f2.  
Here, when f1.backward(), p.d. of f1 w.r.t x1 calculate, store it in x1.grad, same w.r.t x2.  
When f2.backward(), p.d. of f2 w.r.t x1 calculate, add it to existing value in x1.grad, same w.r.t x2.

In [59]:
x1 = torch.tensor(4.0 , requires_grad=True)
x2 = torch.tensor(5.0 , requires_grad=True)
f1 = x1 ** 2 + x2 ** 3
f2 = x2 ** 2 + x1 ** 3

In [60]:
f1.backward()
f2.backward()
print(x1.grad)
print(x2.grad)

tensor(56.)
tensor(85.)


In [61]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

Whole training loop altogether looks like this:

In [62]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(10):
    f = x ** 2
    f.backward()
    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_()

In [63]:
x

tensor(0.5369, requires_grad=True)

## Implementing Linear Regression

### Linear Regression Using Tensors and Autograd

In [114]:
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
df = data.data
target = data.target

In [115]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df, target, test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

Creating tensors for X

In [116]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

Scaling the tensors of X manually

In [117]:
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_train - means) / stds
X_test = (X_test - means) / stds

In [113]:
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

Creating paramters of our linear regression model.

In [118]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

Training model using batch gradient descent.